# M20 · Picking arrivals: STA/LTA triggers, onset times and picks by hand

**Week 3 · about 9.7 h** · before this: M03, M10

> **한국어 요약** 지진파의 도달 시각을 읽는 법을 배웁니다. 자동 피커를 직접 만들고, 두 종류의 필터가 피킹 시각을 어떻게 바꾸는지 잽니다. 이어서 2019년 상주 지진의 P파와 S파를 손으로 피킹하고, 자동 피커들을 여러분의 피킹과 비교합니다.

**You will**

- Build an automatic picker: STA/LTA, triggers, AIC
- Measure how filters move automatic picks
- Pick P and S by hand, with uncertainties and polarities
- Test pickers against your own picks

**New words:** phase pick (위상 피킹, 도달 시각 읽기), onset (온셋, 도달 시작점), pick uncertainty (피킹 불확실성), first-motion polarity (초동 극성), STA/LTA (단기/장기 평균비), trigger (트리거), characteristic function (특성 함수), change point (변화점) · [all terms](../../../docs/glossary.md)

**Optional reading:** Trnkoczy (2012), STA/LTA settings; Bormann (ed.) (2012), reading onsets and polarities

Log any AI help in `work/ai_log/M20.md` ([rules](../../../docs/ai_policy.md)); pass criteria: [criteria/M20.yaml](../../../.sgtl/criteria/M20.yaml).

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019"                    # your STP download of M03
KMA_EVENT_ID = "2019009762"               # the ML 3.9 Sangju earthquake, 21 July 2019 (UTC)
BAND = "HH"                               # broadband seismometers
MAX_EPI_KM = 100.0                        # pick only stations within this epicentral distance
FMIN_HZ, FMAX_HZ = 1.0, 10.0              # band-pass corner frequencies (Hz) of the automatic picker
STA_S, LTA_S = 0.5, 10.0                  # STA and LTA windows (s) to start with
THR_ON, THR_OFF = 4.0, 1.5                # trigger thresholds to start with
AIC_PRE_S, AIC_POST_S = 2.0, 0.5          # AIC window: seconds before and after the trigger
SNR_LEVELS = (5.0, 20.0, 100.0)           # Part 5: signal-to-noise ratios of the made-up records
N_MADE_UP = 40                            # Part 5: made-up records per ratio
PICK_WINDOW_S = (-20.0, 60.0)             # Part 7: part of each record to search (s after the origin)
BAER_CHANGE = ("thr1", 10.0)              # Part 8: the one pk_baer setting you change, and its new value
PICKS_CSV = "../picks/M20_manual_picks.csv"   # your manual picks (work/picks/); M21 reads them too
FIGURE_DIR = "figures"                    # your figures go here, next to this notebook
RESULT_DIR = "results"                    # your result files go here

In [ ]:
import sgtl_tutorial as st
S = st.init("M20")

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import UTCDateTime, read, read_inventory
from obspy.geodetics import gps2dist_azimuth

from sgtl_tutorial import synth_onsets

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)
Path(PICKS_CSV).parent.mkdir(parents=True, exist_ok=True)

## Part 0 · Your data (about 15 min)

The cell finds the ML 3.9 Sangju earthquake in the KMA catalog (times in UTC) and its records in your M03
download. It lists the broadband stations by distance.

In [ ]:
cat = st.data.kma_catalog()
event = cat.loc[cat["event_id"] == KMA_EVENT_ID].iloc[0]
ORIGIN = UTCDateTime(event["origin_utc"].isoformat())
EV_LAT, EV_LON = float(event["latitude"]), float(event["longitude"])
EV_DEPTH_KM, EV_ML = float(event["depth_km"]), float(event["ml"])
print(f"KMA event {KMA_EVENT_ID}: origin {ORIGIN} (UTC), {EV_LAT:.2f} N, {EV_LON:.2f} E, depth {EV_DEPTH_KM:g} km, "
      f"ML {EV_ML:.1f}")

SAC_DIR = st.data.root() / DATASET / "stp_download" / "SAC" / ORIGIN.strftime("%Y%m%d%H%M%S")
if not (SAC_DIR / BAND).is_dir():
    raise FileNotFoundError(f"There are no {BAND} waveforms of this event in {SAC_DIR}. Fetch the {DATASET} data "
                            "with STP first (module M03), or set DATASET in the parameters cell to your folder.")
listed = []
for z_file in sorted((SAC_DIR / BAND).glob(f"*.{BAND}Z.sac")):
    head = read(str(z_file), headonly=True)[0].stats
    epi_km = gps2dist_azimuth(EV_LAT, EV_LON, head.sac.stla, head.sac.stlo)[0] / 1000.0
    listed.append({"station": f"{head.network}.{head.station}", "epi_km": epi_km,
                   "hypo_km": float(np.hypot(epi_km, EV_DEPTH_KM)), "z_file": z_file})
stations = pd.DataFrame(listed).sort_values("epi_km", ignore_index=True)
print(f"{len(stations)} stations with {BAND}Z records; {int((stations['epi_km'] <= MAX_EPI_KM).sum())} within "
      f"{MAX_EPI_KM:g} km of the epicenter")
print(stations[["station", "epi_km", "hypo_km"]].round(1).to_string(index=False))

NEAREST = stations.loc[0, "station"]
near = read(str(SAC_DIR / BAND / f"*.{NEAREST}.{BAND}?.sac"))
near.sort(keys=["channel"])
near_z = near.select(component="Z")[0]
print(near)

## Part 1 · What a pick is (about 30 min)

A **phase pick** is the arrival time of one phase (P or S) at one station, with three parts:

- the **onset**: the first sample where the new wave shows;
- a **pick uncertainty**: how far the true onset may lie from your pick, in seconds;
- for P, the **first-motion polarity**: up (`U`), down (`D`) or unclear (`?`).

The figure shows the nearest station, then one second around its P wave.

In [ ]:
shown = near.copy().detrend("demean")
fig, axes = plt.subplots(4, 1, figsize=(7.5, 7.0))
for ax, tr in zip(axes[:3], shown):
    t_rel = tr.times(reftime=ORIGIN)
    keep = (t_rel >= -2.0) & (t_rel <= 12.0)
    ax.plot(t_rel[keep], tr.data[keep], color="k", lw=0.5)
    ax.set_ylabel(f"{tr.stats.channel} (counts)")
axes[2].set_xlabel("Time after origin (s)")
z_rel = shown.select(component="Z")[0]
t_rel = z_rel.times(reftime=ORIGIN)
quiet = (t_rel > -25.0) & (t_rel < -5.0)
noise_counts = float(np.std(z_rel.data[quiet]))
peak_counts = float(np.abs(z_rel.data[(t_rel > 0.0) & (t_rel < 12.0)]).max())
first_big = float(t_rel[np.flatnonzero((np.abs(z_rel.data) > 20.0 * noise_counts) & (t_rel > -5.0))[0]])
zoom = (t_rel > first_big - 0.6) & (t_rel < first_big + 0.4)
axes[3].plot(t_rel[zoom], z_rel.data[zoom], "k.-", lw=0.6, ms=2)
axes[3].set_ylim(-30.0 * noise_counts, 30.0 * noise_counts)
axes[3].set_xlabel("Time after origin (s); every dot is one sample")
axes[3].set_ylabel(f"{z_rel.stats.channel} (counts)")
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "nearest_station_first_seconds.png", dpi=150)
print(f"{NEAREST}: noise {noise_counts:.0f} counts (standard deviation 25-5 s before the origin); largest vertical "
      f"swing in the first 12 s {peak_counts:.0f} counts, about {peak_counts / noise_counts:.0f} times the noise")

Here the P wave leaves the noise within one or two samples: an **impulsive onset**. An **emergent onset** grows
slowly out of the noise and is harder to time. Most onsets are less clear than this one.

**Watch out:** pick only the direct P and S within about 100 km. Farther away, a head wave can arrive first
(M30); never pick it for a location.

## Part 2 · The STA/LTA ratio (about 30 min, then task 1)

Allen's STA/LTA detector squares the samples: this **characteristic function** follows the energy of the
shaking. It averages the squares over two windows that end at the current sample: a short one (STA) and a long
one (LTA). Their ratio, **STA/LTA**, stays near 1 in noise and jumps when an arrival comes in, because the short
window fills first. Before the long window is full, the ratio is 0.

### M20-T1 · An STA/LTA ratio · core · about 60 min

Write `sta_lta(data, nsta, nlta)`. It returns the STA/LTA ratio at every sample; the docstring gives the details.

Example: `data = [1, 1, 1, 1, 2, 2]`, `nsta = 2`, `nlta = 4` gives `[0, 0, 0, 1.0, 1.4286, 1.6]` (rounded).

Not allowed: ObsPy's trigger functions (`obspy.signal.trigger`, `Trace.trigger`) and pyrocko.

In [ ]:
# Before you code: what does numpy give when you square a 32-bit integer of 100,000?
st.predict("M20-T1", """

""")

In [ ]:
def sta_lta(data, nsta, nlta):
    """STA/LTA ratio of the squared samples; both windows end at the current sample.

    data       : 1-D sequence of numbers: a list, or a numpy array of any number type (32-bit integers too)
    nsta, nlta : window lengths in samples (whole numbers), 1 <= nsta < nlta <= len(data)
    returns    : numpy float64 array as long as data, with ratio[i] = STA[i] / LTA[i]. STA[i] is the mean of
                 data[j] ** 2 over the nsta samples j = i - nsta + 1 ... i; LTA[i] the same over nlta samples.
                 The ratio is 0 before sample nlta - 1 (the long window is not full) and wherever the LTA is 0.
    """
    # Step 1: a float64 array of the squared samples (the characteristic function).
    # Step 2: the mean over the last nsta and over the last nlta samples, at every sample from nlta - 1 on. A fast
    #         way: keep a running total of the squares; the sum over a window is the difference of two totals.
    # Step 3: divide where the LTA is above 0; leave 0 everywhere else.
    # YOUR CODE HERE
    raise NotImplementedError("M20-T1: write your code here")

In [ ]:
st.check("M20-T1", func=sta_lta)

The check also used 32-bit records with large values. Compare with your prediction:

In [ ]:
counts_32 = np.array([30_000, 100_000, 300_000], dtype=np.int32)
print("int32 counts squared:  ", counts_32 * counts_32)
print("float64 counts squared:", counts_32.astype(np.float64) ** 2)

**Watch out:** a 32-bit integer holds at most about 2.1 billion, so a larger square silently wraps around to a
wrong number. Convert counts to 64-bit floats (`np.float64`) before you square them.

Your function on a made-up record from `synth_onsets`, the course's practice-record maker. The dashed line marks
the true onset.

In [ ]:
demo_trace, demo_truth = synth_onsets.onset_trace(np.random.default_rng(20), snr=8.0)
demo_rate = demo_trace.stats.sampling_rate
demo_ratio = sta_lta(demo_trace.data, int(STA_S * demo_rate), int(LTA_S * demo_rate))
t_demo = demo_trace.times()
fig, (ax_rec, ax_ratio) = plt.subplots(2, 1, figsize=(7.5, 4.6), sharex=True)
ax_rec.plot(t_demo, demo_trace.data, color="k", lw=0.5)
ax_rec.set_ylabel("Made-up record")
ax_ratio.plot(t_demo, demo_ratio, color="k", lw=0.8)
ax_ratio.set_ylabel("STA/LTA")
ax_ratio.set_xlabel("Time after the first sample (s); dashed: the true onset")
for ax in (ax_rec, ax_ratio):
    ax.axvline(demo_truth["onset_s"], color="tab:red", lw=0.8, ls="--")
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "sta_lta_made_up.png", dpi=150)
after = t_demo >= demo_truth["onset_s"]
print(f"true onset {demo_truth['onset_s']:.2f} s; largest ratio {demo_ratio.max():.1f} at "
      f"{t_demo[np.argmax(demo_ratio)]:.2f} s; largest ratio before the onset {demo_ratio[~after].max():.2f}")

**Explain:** In your figure, how long after the true onset does the ratio peak, and what sets that delay?

## Part 3 · Triggers: on and off (about 20 min, then task 2)

A **trigger** is the moment a detector declares that an arrival has started. A ratio that wobbles around one
threshold would switch on and off many times. So the trigger turns on when the ratio rises above `thr_on`, and
off only when it falls to `thr_off`, a lower value. A trigger on noise is a **false trigger**.

### M20-T2 · Trigger on and off · core · about 40 min

Write `trigger_windows(cft, thr_on, thr_off)`. It returns the `(on, off)` sample pairs of all triggers, in time
order; the docstring gives the rules.

Example: `cft = [0, 5, 5, 2, 1, 0, 5, 0]`, `thr_on = 4`, `thr_off = 1.5` gives `[(1, 3), (6, 6)]`.

- "Above" is strict: a value equal to `thr_on` does not turn a trigger on; one equal to `thr_off` turns it off.
- Not allowed: ObsPy's trigger functions and pyrocko.

In [ ]:
# Before you code: how many triggers does cft = [0, 5, 3, 5, 1, 5] give with thr_on = 4 and thr_off = 1.5?
st.predict("M20-T2", """

""")

In [ ]:
def trigger_windows(cft, thr_on, thr_off):
    """(on, off) sample pairs where cft first rises above thr_on and stays above thr_off.

    cft             : 1-D sequence of numbers (for example an STA/LTA ratio)
    thr_on, thr_off : the on and off thresholds, thr_off < thr_on
    returns         : list of (on, off) pairs of int sample indices, in time order ([] when nothing triggers).
                      on is the first sample greater than thr_on; off is the last sample still greater than
                      thr_off (the last sample of cft when the trigger is still on there). After off, a new
                      trigger needs a value greater than thr_on again.
    """
    # Step 1: walk through the samples, remembering whether a trigger is on and where it started.
    # Step 2: with no trigger on, start one at a value > thr_on; with one on, end it at a value <= thr_off
    #         (its off is the sample before).
    # Step 3: a trigger still on at the end closes at the last sample.
    # YOUR CODE HERE
    raise NotImplementedError("M20-T2: write your code here")

In [ ]:
st.check("M20-T2", func=trigger_windows)

Your two functions on the nearest station's vertical record, as recorded and band-passed (causal). Times are
seconds after the origin.

In [ ]:
for label, record in (("as recorded", near_z.copy().detrend("demean")),
                      ("band-passed", near_z.copy().detrend("demean").taper(0.05, type="hann", max_length=5.0).filter(
                          "bandpass", freqmin=FMIN_HZ, freqmax=FMAX_HZ, corners=4, zerophase=False))):
    rate = record.stats.sampling_rate
    windows = trigger_windows(sta_lta(record.data, int(STA_S * rate), int(LTA_S * rate)), THR_ON, THR_OFF)
    starts = [round(float(record.times(reftime=ORIGIN)[on]), 2) for on, _ in windows]
    print(f"{label:>12s}: {len(windows)} triggers; on at {starts[:8]} s after the origin"
          + (" ..." if len(starts) > 8 else ""))

**Explain:** Which trigger is the P wave, and why does the record give many more triggers without the
band-pass?

**Optional: explorer E3.** Sliders for the windows, the thresholds and the band-pass redraw your own `sta_lta`
and `trigger_windows` on the nearest station. Which setting gives one trigger for P and none in the noise?

In [ ]:
# Optional explorer E3 (not checked): your STA/LTA and triggers on the nearest station's vertical record.
st.widgets.stalta_tuner(near_z, sta_lta=sta_lta, trigger=trigger_windows, reftime=ORIGIN, window=(-25.0, 60.0),
                        fmin=FMIN_HZ, fmax=FMAX_HZ, sta_s=STA_S, lta_s=LTA_S, thr_on=THR_ON, thr_off=THR_OFF,
                        taper=0.01, xlabel="Time after origin (s)")

## Part 4 · Timing an onset: the AIC (about 15 min, then task 3)

A trigger says that something arrived, not exactly when. To time the onset, split a short window of $N$ samples
into noise before and signal after. Maeda's method scores each split $k$ with the **Akaike information
criterion** (AIC); a lower score means a better split:

$$\mathrm{AIC}(k) = k \ln \operatorname{var}(x_1 \dots x_k) + (N - k - 1) \ln \operatorname{var}(x_{k+1} \dots x_N)$$

The split with the smallest AIC is the onset: the **change point** where the record changes.

### M20-T3 · The AIC onset · core · about 45 min

Write `aic_onset(data)`. It returns the `k` with the smallest AIC, for `k` from 2 to `len(data) - 2`.

Example: `[1, -1, 1, -1, 1, 6, -6, 6, -6, 6]` gives `5`.

- Use `np.var` and the natural log; on a tie, take the smallest `k`.
- Not allowed: ObsPy's trigger functions (they include an AIC) and pyrocko.

In [ ]:
# Before you code: noise grows 10 times at sample 250 of 400. Sketch AIC(k): where is it smallest?
st.predict("M20-T3", """

""")

In [ ]:
def aic_onset(data):
    """First sample of the second part of a short record, by the Akaike information criterion (Maeda 1985).

    data    : 1-D sequence of at least 5 numbers
    returns : Python int k, 2 <= k <= len(data) - 2, that minimizes
              AIC(k) = k ln var(data[:k]) + (N - k - 1) ln var(data[k:]), with N = len(data), var = np.var
              (it divides by the number of samples) and ln the natural log; the smallest such k on a tie
    """
    # Step 1: a float64 array and its length N.
    # Step 2: the AIC for every split k from 2 to N - 2 (both parts keep at least two samples).
    # Step 3: the k of the smallest AIC (the first one on a tie), as a Python int.
    # YOUR CODE HERE
    raise NotImplementedError("M20-T3: write your code here")

In [ ]:
st.check("M20-T3", func=aic_onset)

Your function on a made-up record, in four windows around the true onset. The error is your onset minus the true
one, in samples (0.01 s each).

In [ ]:
aic_trace, aic_truth = synth_onsets.onset_trace(np.random.default_rng(31), snr=10.0)
true_k = aic_truth["onset_index"]
for before_s, after_s in ((2.0, 0.5), (2.0, 5.0), (0.3, 0.5), (8.0, 0.5)):
    lo = true_k - int(before_s * 100)
    hi = true_k + int(after_s * 100)
    found = lo + aic_onset(aic_trace.data[lo:hi + 1])
    print(f"window from {before_s:3.1f} s before to {after_s:3.1f} s after the onset: "
          f"error {found - true_k:+4d} samples")

**Explain:** Why does the window with 8 s of noise fail, while the three shorter windows work?

## Part 5 · An automatic picker on a filtered record (about 15 min, then task 4)

Real records need a band-pass filter before picking, or slow ocean noise near 0.2 Hz rules the ratio. From
M10: a causal filter uses only past samples; a zero-phase filter runs forward, then backward. Task 4 joins your
three functions into one picker, and you test it with both kinds of filter.

### M20-T4 · An automatic picker on a filtered record · core · about 75 min

Write `auto_pick(tr, fmin, fmax, zerophase, sta_s=0.5, lta_s=10.0, thr_on=4.0, thr_off=1.5, pre_s=2.0,
post_s=0.5)`. It band-passes a copy of `tr`, finds the first trigger with your `sta_lta` and `trigger_windows`,
and times the onset with your `aic_onset`. It returns a `UTCDateTime`, or `None` without a trigger.

Follow the five steps of the docstring exactly: the check repeats them with ObsPy. Not allowed: ObsPy's trigger
functions and pyrocko.

In [ ]:
# Before you code: with a causal and with a zero-phase band-pass, will your picks come before, at or after
# the true onset?
st.predict("M20-T4", """

""")

In [ ]:
def auto_pick(tr, fmin, fmax, zerophase, sta_s=0.5, lta_s=10.0, thr_on=4.0, thr_off=1.5, pre_s=2.0, post_s=0.5):
    """Automatic onset: your STA/LTA trigger, then your AIC onset.

    tr               : ObsPy Trace (never changed)
    fmin, fmax       : band-pass corner frequencies (Hz)
    zerophase        : True: zero-phase band-pass; False: causal
    sta_s, lta_s     : STA and LTA window lengths (s)
    thr_on, thr_off  : trigger thresholds
    pre_s, post_s    : AIC window: seconds before and after the first trigger-on sample
    returns          : UTCDateTime of the onset, or None

    The five steps:
    1. A copy of tr: detrend("demean"), taper(max_percentage=0.05, type="hann"), then
       filter("bandpass", freqmin=fmin, freqmax=fmax, corners=4, zerophase=zerophase).
    2. nsta = round(sta_s * rate), nlta = round(lta_s * rate), with rate the sampling rate.
    3. Your sta_lta on the band-passed samples, then your trigger_windows(ratio, thr_on, thr_off).
       No trigger: return None.
    4. With i the on sample of the first trigger and N the number of samples: the AIC window runs from
       a = max(0, i - round(pre_s * rate)) to b = min(N - 1, i + round(post_s * rate)), both included.
       The onset is k = a + aic_onset(band-passed samples a to b).
    5. Return tr.stats.starttime + k * tr.stats.delta.
    """
    # Step 1: a band-passed copy (mean removed, 5 % Hann taper, corners=4, zero-phase or causal as asked).
    # Step 2: the window lengths in samples.
    # Step 3: your ratio and your triggers; None when there is no trigger.
    # Step 4: the AIC window around the first on sample (clipped to the record); your AIC onset inside it.
    # Step 5: the moment of that sample.
    # YOUR CODE HERE
    raise NotImplementedError("M20-T4: write your code here")

In [ ]:
st.check("M20-T4", func=auto_pick)

Now measure. For each signal-to-noise ratio (SNR) in `SNR_LEVELS`, the cell picks `N_MADE_UP` made-up records
with a known onset, causal and zero-phase. It prints your pick minus the true onset, in samples: the median and
the 10 % and 90 % points. It takes about half a minute.

In [ ]:
measured = []
examples = {}
for snr in SNR_LEVELS:
    made_up = synth_onsets.onset_set(np.random.default_rng(int(snr)), N_MADE_UP, snr=snr)
    for zerophase in (False, True):
        late = []
        for trace, truth in made_up:
            pick = auto_pick(trace, FMIN_HZ, FMAX_HZ, zerophase, STA_S, LTA_S, THR_ON, THR_OFF, AIC_PRE_S, AIC_POST_S)
            if pick is not None:
                late.append((pick - truth["onset"]) * trace.stats.sampling_rate)
        late = np.round(np.array(late))
        examples[(snr, zerophase)] = (made_up[0], auto_pick(made_up[0][0], FMIN_HZ, FMAX_HZ, zerophase, STA_S, LTA_S,
                                                             THR_ON, THR_OFF, AIC_PRE_S, AIC_POST_S))
        measured.append({"SNR": snr, "filter": "zero-phase" if zerophase else "causal",
                         "picked": f"{late.size}/{len(made_up)}", "median": np.median(late),
                         "10 %": np.percentile(late, 10), "90 %": np.percentile(late, 90)})
shift_table = pd.DataFrame(measured)
print("pick minus true onset, in samples (100 samples per second):")
print(shift_table.to_string(index=False))

(example_trace, example_truth), _ = examples[(SNR_LEVELS[-1], False)]
fig, ax = plt.subplots(figsize=(7.5, 3.4))
offset = 0.0
for zerophase, color in ((False, "tab:blue"), (True, "tab:red")):
    version = example_trace.copy().detrend("demean").taper(0.05, type="hann").filter(
        "bandpass", freqmin=FMIN_HZ, freqmax=FMAX_HZ, corners=4, zerophase=zerophase)
    t_rel = version.times() - example_truth["onset_s"]
    keep = (t_rel > -0.6) & (t_rel < 0.4)
    scale = np.abs(version.data[keep]).max()
    ax.plot(t_rel[keep], version.data[keep] / scale + offset, color=color, lw=0.8,
            label="zero-phase" if zerophase else "causal")
    pick = examples[(SNR_LEVELS[-1], zerophase)][1]
    if pick is not None:
        ax.plot(pick - example_truth["onset"], offset, "v", color=color, ms=7)
    offset += 2.2
ax.axvline(0.0, color="k", lw=0.8, ls="--")
ax.set_xlabel(f"Time after the true onset (s); SNR {SNR_LEVELS[-1]:g}; triangles: your picks")
ax.set_yticks([])
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "picks_causal_zero_phase.png", dpi=150)

**Explain:** How do the zero-phase errors change with the SNR, and why is that a problem for a network?

## Part 6 · Picking by hand (about 20 min, then task 5)

Now the real records. Part 7, M21 and M31 all use your picks, so take your time.

First, the instrument (M03). A negative product of the stage gains would mark a channel that records the ground
upside down. The cell prints three signs per vertical channel: first stage, all stages, stored sensitivity.
The metadata are StationMetadata by Byeong-Seok Ahn (Pukyong National University; Ahn, Kang & Jung 2024): cite
them when you use them.

In [ ]:
META_DIR = st.data.root() / "metadata" / "stationxml"


def sign_of(value):
    """'+' or '-' for a number."""
    return "+" if value > 0 else "-"


signs = []
for row in stations.loc[stations["epi_km"] <= MAX_EPI_KM].itertuples():
    line = {"station": row.station, "epi_km": round(row.epi_km, 1)}
    try:
        inventory = read_inventory(str(META_DIR / f"{row.station}.xml"))
        channel = inventory.select(channel=f"{BAND}Z", time=ORIGIN)[0][0][0]
        gains = [stage.stage_gain for stage in channel.response.response_stages]
        line.update({"first stage": sign_of(gains[0]), "all stages": sign_of(np.prod(gains)),
                     "stored sensitivity": sign_of(channel.response.instrument_sensitivity.value)})
    except Exception as exc:  # missing file, or no channel epoch at this time
        line.update({"first stage": f"unknown ({type(exc).__name__})", "all stages": "", "stored sensitivity": ""})
    signs.append(line)
signs = pd.DataFrame(signs)
print(f"Signs of the {BAND}Z response at the {len(signs)} stations within {MAX_EPI_KM:g} km (StationMetadata):")
print(signs.to_string(index=False))
odd = signs.loc[(signs["all stages"] == "-") & (signs["stored sensitivity"] == "+"), "station"]
print(f"\n{len(odd)} of {len(signs)} stations: the stages multiply to a negative number, but the stored sensitivity "
      f"is positive: {', '.join(odd) or 'none'}")

At several stations the file disagrees with itself: the stages say "reversed", the stored sensitivity says "not
reversed". Only the records can decide; you test this with your own picks after task 5.

**Watch out:** never flip a polarity because of a gain sign. Write each polarity as you see it in the counts.

**Two tools.** Pick in SeisComP as the [GUI workshop](../../../docs/seiscomp_gui.md) shows, then import the picks
into `PICKS_CSV`. Or use the notebook picker E2 below: its Save button writes the same file. Keep one reviewed
file. Neither tool picks for you.

**Watch out:** zero-phase filters move onsets early (task 4). Pick, and read polarities, only on causal or
unfiltered records.

**How to pick.** Start with the nearest stations.

- Zoom in until every sample shows.
- P: on the vertical (Z) record, at the last noise sample before the arrival. The polarity is the sign of the
  first swing; write `?` if the noise is as large.
- S: on a horizontal (N or E) record, where size and rhythm change.
- Uncertainty: half the range of times you would accept; hundredths of a second for a sharp P.

### M20-T5 · Picking P and S by hand · core · about 120 min

Pick P and S at 6 or more stations within `MAX_EPI_KM` (100) km and save them to `PICKS_CSV`. Each station gets
one P (vertical channel, with its polarity) and one S (horizontal channel). Uncertainties: P 0.005-0.5 s, S
0.01-1.0 s. At least 3 P picks need `U` or `D`.

The check also tests that each record changes at your picks and that all S-P times fit one earthquake.

In [ ]:
# Before you pick: with VP about 6 km/s and VP/VS about 1.73, when do P and S reach the nearest station, 18 km
# from the hypocenter?
st.predict("M20-T5", """

""")

In [ ]:
# Alternative notebook picker E2: one station at a time. If you imported SeisComP picks,
# it reads that file, but you do not need to press its Save button or use it to pick again.
picker = st.widgets.phase_picker(SAC_DIR / BAND, origin=ORIGIN, epicenter=(EV_LAT, EV_LON), max_km=MAX_EPI_KM,
                                 csv_path=PICKS_CSV, window=(-5.0, 30.0), fmin=FMIN_HZ, fmax=FMAX_HZ)
picker

In [ ]:
# The file has one row per pick, with the columns station (NET.STA), phase (P or S), channel (P on a channel
# ending in Z, S on one ending in N or E), time_utc (ISO text in UTC), uncertainty_s and polarity (P: U, D or ?;
# S: ?). At every station S comes after P. E2's Save button and the SeisComP import both write this format.
# SeisComP route: import your saved XML in the server terminal as the GUI workshop shows;
# in this cell verify that Path(PICKS_CSV).is_file(), then run the unchanged check below.
# E2 route: pick in the picker above and press Save. You can also type picks, one call per pick:
#     picker.set_pick("KS.XXXX", "P", <seconds after the origin>, <uncertainty in s>, "U", channel="HHZ")
# then save with picker.save(). Replace the two lines below with the calls you used (at least picker.save()).
# YOUR CODE HERE
raise NotImplementedError("M20-T5: write your code here")

In [ ]:
st.check("M20-T5", answer=PICKS_CSV, dataset=DATASET)

Your picks with S-P times and distances, and S-P against distance. For the nearest station the cell also prints
the rough onset of Part 1 next to your P pick.

In [ ]:
mine = pd.read_csv(PICKS_CSV)
both = mine.pivot_table(index="station", columns="phase", values="time_utc", aggfunc="first")
table = stations.set_index("station").loc[both.index, ["epi_km", "hypo_km"]].copy()
table["P_s"] = [UTCDateTime(t) - ORIGIN for t in both["P"]]
table["S_s"] = [UTCDateTime(t) - ORIGIN for t in both["S"]]
table["S-P_s"] = table["S_s"] - table["P_s"]
table = table.sort_values("hypo_km")
print(table.round(3).to_string())
nearest_p = f"your P pick {table.loc[NEAREST, 'P_s']:.2f} s" if NEAREST in table.index else "not picked"
print(f"\n{NEAREST} (nearest station): first sample above 20 times the noise {first_big:.2f} s after the origin; "
      f"{nearest_p}")
fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.plot(table["hypo_km"], table["S-P_s"], "o", color="k", ms=5)
label_offsets = [(6, 8), (6, -15), (-34, 20), (-34, -26)]
for i, (name, row) in enumerate(table.iterrows()):
    ax.annotate(name, (row["hypo_km"], row["S-P_s"]), xytext=label_offsets[i % len(label_offsets)],
                textcoords="offset points", fontsize=8, arrowprops={"arrowstyle": "-", "color": "0.6", "lw": 0.5})
ax.margins(x=0.10, y=0.15)
ax.set_xlabel("Hypocentral distance from the KMA hypocenter (km)")
ax.set_ylabel("S-P time (s)")
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "manual_s_minus_p.png", dpi=150)

Your polarities next to the metadata signs, sorted by **azimuth** (the direction from the epicenter to the
station, clockwise from north). Stations in nearly the same direction usually move first the same way.

In [ ]:
first_gain = signs.set_index("station")["first stage"]
p_polarity = mine.loc[mine["phase"] == "P"].set_index("station")["polarity"]
facing = []
for name in p_polarity.index:
    head = read(str(stations.set_index("station").loc[name, "z_file"]), headonly=True)[0].stats
    _, azimuth_deg, _ = gps2dist_azimuth(EV_LAT, EV_LON, head.sac.stla, head.sac.stlo)
    facing.append({"station": name, "azimuth_deg": round(azimuth_deg), "epi_km": round(table.loc[name, "epi_km"], 1),
                   "your polarity": p_polarity[name], "first-stage gain": first_gain.get(name, "unknown")})
facing = pd.DataFrame(facing).sort_values("azimuth_deg", ignore_index=True)
print(facing.to_string(index=False))

**Explain:** Which stations with a negative first-stage gain can your picks test against a neighbor at a similar
azimuth?

## Part 7 · Your picker against your own picks (about 20 min, then task 6)

Your manual picks are an independent reference for your picker. A few large errors (a false trigger, a missed
station) should not hide how well the rest agree. So use the median error and the **median absolute error** (the
median of the error sizes), not the mean.

### M20-T6 · Your picker against your own picks · core · about 90 min

Run your `auto_pick` on the HHZ record of every station in your picks file, for a grid of 4 settings or more
(at least two `sta_s` and two `thr_on` values). Measure its errors against your manual P picks, choose one setting, and write the JSON
file that the code-cell comments describe. Then set `auto_result = {"json": <path>, "picker": auto_pick}`.

The check reruns your `auto_pick` on your records.

In [ ]:
# Before you code: with the start settings (STA 0.5 s, thr_on 4), at how many of your stations will the first
# trigger be the P wave?
st.predict("M20-T6", """

""")

In [ ]:
# Read your picks and the HHZ records; run auto_pick for every setting of your grid; compute the errors and the
# statistics; choose a setting; write the JSON file; set auto_result.
# Records: the HHZ file of each station with a P pick in your file, cut with
#     read(path)[0].trim(ORIGIN + PICK_WINDOW_S[0], ORIGIN + PICK_WINDOW_S[1]).
# Picker: your auto_pick with FMIN_HZ, FMAX_HZ, zerophase=False, then the setting, then AIC_PRE_S, AIC_POST_S.
# Error at a station: automatic minus manual P time (s; positive = the picker is late). No trigger = missed.
# Statistics of a setting, over the picked stations: n_picked, median_error_s, median_abs_error_s (medians, not
# means) and max_abs_error_s (the largest absolute error).
# JSON keys: picks_csv (the path of your picks file), band_hz [fmin, fmax], window_s (PICK_WINDOW_S), aic_s
# [pre_s, post_s], grid (one dict per setting: sta_s, lta_s, thr_on, thr_off and the four statistics), chosen
# (the grid entry you choose) and rows (one dict per station of your file, for the chosen setting: station,
# sac_path, manual_p_utc, auto_p_utc, error_s; the last two null when missed). Times as ISO text in UTC.
# YOUR CODE HERE
raise NotImplementedError("M20-T6: write your code here")

In [ ]:
st.check("M20-T6", answer=auto_result, dataset=DATASET)

**Explain:** Which setting did you choose, and what in your grid table supports that choice?

## Part 8 · A production picker: unchanged, then one change (about 30 min)

ObsPy's `pk_baer` (Baer and Kradolfer) is used at many observatories. It picks the first trigger that lasts long
enough, with no AIC step. Its codes, such as `IPU0`, mean impulsive (I) or emergent (E), P, up (U) or down (D),
and a quality from 0 (best) to 4. You run it on the records of task 6 three times: as documented, unchanged
again, then with one setting changed.

Its settings, in samples (100 samples are 1 s):

| Setting | Meaning |
|---|---|
| `thr1` | trigger threshold, in standard deviations of the noise |
| `thr2` | samples below it update the noise level |
| `tdownmax` | how long the function may dip below `thr1` |
| `tupevent` | how long a trigger must last |
| `preset_len` | samples that set the picker up |
| `p_dur` | samples that set the quality |

In [ ]:
from obspy.signal.trigger import pk_baer

BAER_SETTINGS = {"tdownmax": 20, "tupevent": 60, "thr1": 7.0, "thr2": 12.0, "preset_len": 100, "p_dur": 100}
p_rows = mine.loc[mine["phase"] == "P"].set_index("station")
cut_records = {}
for name in p_rows.index:
    z_path = next((SAC_DIR / BAND).glob(f"*.{name}.{BAND}Z.sac"))
    record = read(str(z_path))[0].trim(ORIGIN + PICK_WINDOW_S[0], ORIGIN + PICK_WINDOW_S[1])
    cut_records[name] = record.detrend("demean")


def run_baer(settings):
    """pk_baer with these settings on every cut record: its pick minus your P pick (s), and its code."""
    lines = []
    for name, record in cut_records.items():
        sample, code = pk_baer(record.data, record.stats.sampling_rate, **settings)
        picked = record.stats.starttime + sample * record.stats.delta
        error = round(float(picked - UTCDateTime(p_rows.loc[name, "time_utc"])), 3) if code else None
        lines.append({"station": name, "your polarity": p_rows.loc[name, "polarity"], "error_s": error,
                      "code": code or "no pick"})
    return pd.DataFrame(lines)


def agreement(result):
    """How many picks lie within 0.1 s of yours, and at how many of those the U/D letter is your polarity."""
    close = pd.to_numeric(result["error_s"]).abs() <= 0.1
    same = close & (result["code"].str[2] == result["your polarity"])
    return (f"{int(close.sum())} of {len(result)} picks within 0.1 s of yours; "
            f"the U/D letter agrees with your polarity at {int(same.sum())} of them")


baer_first = run_baer(BAER_SETTINGS)
print(baer_first.to_string(index=False))
print(agreement(baer_first))

**An unchanged rerun.** Is `pk_baer` **deterministic**: does it give exactly the same result when nothing
changed? If not, every later comparison mixes your change with that scatter.

In [ ]:
baer_rerun = run_baer(BAER_SETTINGS)
print("rerun identical to the first run:", baer_rerun.equals(baer_first))

**One setting changed.** `BAER_CHANGE` (parameters cell) names the setting and its new value. To try another,
edit it there, then rerun the parameters cell and this cell.

In [ ]:
changed_name, changed_value = BAER_CHANGE
if changed_name not in BAER_SETTINGS:
    raise KeyError(f"BAER_CHANGE must name one of {list(BAER_SETTINGS)}, not {changed_name!r}")
baer_changed = run_baer(dict(BAER_SETTINGS, **{changed_name: changed_value}))
print(f"changed: {changed_name} from {BAER_SETTINGS[changed_name]} to {changed_value}; all other settings unchanged")
side_by_side = baer_first.merge(baer_changed, on=["station", "your polarity"], suffixes=(" unchanged", " changed"))
print(side_by_side.to_string(index=False))
print("unchanged:", agreement(baer_first))
print("changed:  ", agreement(baer_changed))

**Explain (not checked):** What did your one change fix or break, compared with the unchanged run?

## Wrap-up

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M20")

**Optional:**

1. Repeat Part 5 with emergent onsets: `synth_onsets.onset_set(rng, 40, snr=20.0, rise_s=0.3)`.
2. Pick the HG (accelerometer) records of your stations. Do your picks agree?

**Read more:** Trnkoczy (2012), STA/LTA; Maeda (1985), the AIC onset; Diehl et al. (2009), pick uncertainties.
The [reading list](../../../docs/reading_list.md) has the rest.

Next: M21 runs deep-learning pickers on these records.